# 01 - Análise Exploratória de Dados

Objetivo deste notebook: importar, ler e entender o conjunto de dados Instacart Market Basket Analysis antes de qualquer modelagem.

Nesta etapa vamos responder:

- Do que se trata a base de dados?
- Quais arquivos, colunas e relacionamentos existem?
- Qual é a qualidade inicial das variáveis?
- Existem nulos, duplicados ou problemas de integridade?
- Quais sinais parecem úteis para um sistema de recomendação?

## 1. Configuração inicial

Nesta etapa são carregadas as bibliotecas utilizadas no notebook, definidos os caminhos principais do projeto e configuradas as opções de exibição do pandas. Também importamos as classes do `ml_prep_kit`, que centralizam tarefas reutilizáveis como leitura de CSVs e validação de dados. Essa estratégia evita duplicação de código no notebook e mantém a análise focada nas perguntas de negócio.


In [3]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import sys

import pandas as pd

# Define caminhos de forma flexível para rodar o notebook da raiz ou da pasta notebooks.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "instacart"
ML_PREP_KIT_SRC = PROJECT_ROOT / "ml_prep_kit" / "src"

# Usa o ml_prep_kit como camada reutilizável para leitura e validação dos dados.
if str(ML_PREP_KIT_SRC) not in sys.path:
    sys.path.insert(0, str(ML_PREP_KIT_SRC))

from ml_prep_kit import CSVDataLoader, DataValidator

# Configura a exibição para facilitar a leitura das tabelas no notebook.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.2f}".format)

validator = DataValidator()


### Definição dos caminhos

O caminho dos dados brutos é centralizado em `RAW_DATA_DIR`. Essa definição evita caminhos fixos espalhados pelo notebook e facilita a reprodução da análise em outro ambiente.


In [5]:
# Exibe o diretório usado para carregar os arquivos brutos do Instacart.
RAW_DATA_DIR


PosixPath('/Users/cassiojr/FIAP/Tech Challenge 2/FIAP-fase2-e-commerce/data/raw/instacart')

## 2. Arquivos disponíveis

A base é composta por tabelas relacionais. Os arquivos maiores são os itens comprados em pedidos anteriores e a tabela de pedidos.

In [5]:
# Lista os arquivos brutos e seus tamanhos para entender o volume inicial da base.
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

file_inventory = pd.DataFrame(
    {
        "file_name": [file.name for file in csv_files],
        "size_mb": [file.stat().st_size / 1024**2 for file in csv_files],
    }
).sort_values("size_mb", ascending=False)

file_inventory

,file_name,size_mb
2,order_products__prior.csv,550.80
4,orders.csv,103.92
3,order_products__train.csv,23.54
5,products.csv,2.07
0,aisles.csv,0.00
1,departments.csv,0.00


## 3. Leitura dos dados

Os arquivos CSV são carregados em DataFrames separados, respeitando a estrutura original da base. A leitura usa a classe `CSVDataLoader`, do `ml_prep_kit`, para centralizar o carregamento dos arquivos e evitar chamadas repetidas de `pd.read_csv` no notebook.


In [23]:
# Carrega as tabelas brutas usando a classe reutilizável do ml_prep_kit.
loader = CSVDataLoader(RAW_DATA_DIR)

datasets = loader.load(
    {
        "aisles": "aisles.csv",
        "departments": "departments.csv",
        "products": "products.csv",
        "orders": "orders.csv",
        "order_products_train": "order_products__train.csv",
        "order_products_prior": "order_products__prior.csv",
    }
)

# Mantém variáveis individuais para deixar as análises seguintes mais legíveis.
aisles = datasets["aisles"]
departments = datasets["departments"]
products = datasets["products"]
orders = datasets["orders"]
order_products_train = datasets["order_products_train"]
order_products_prior = datasets["order_products_prior"]


### Visão geral dos volumes

A tabela abaixo resume o tamanho de cada DataFrame em número de linhas, colunas e memória utilizada. Esse passo ajuda a identificar quais tabelas exigem mais atenção em joins, agregações e etapas futuras de modelagem.

In [24]:
# Resume volume, quantidade de colunas e uso de memória de cada tabela.
overview = validator.summarize(datasets)

overview


,dataset,rows,columns,memory_mb
5,order_products_prior,32434489,4,989.82
3,orders,3421083,7,332.71
4,order_products_train,1384617,4,42.26
2,products,49688,4,4.93
0,aisles,134,2,0.01
1,departments,21,2,0.00


## 4. Amostra e esquema dos dados

Nesta parte vamos observar colunas, tipos de dados e primeiras linhas para entender a função de cada tabela.

In [25]:
# Gera um resumo de tipos, nulos e cardinalidade usando o validador reutilizável.
schema = validator.describe_schema(datasets)

schema


,dataset,column,dtype,nulls,null_rate,unique_values
0,aisles,aisle_id,int64,0,0.00,134
1,aisles,aisle,str,0,0.00,134
2,departments,department_id,int64,0,0.00,21
3,departments,department,str,0,0.00,21
4,products,product_id,int64,0,0.00,49688
5,products,product_name,str,0,0.00,49688
6,products,aisle_id,int64,0,0.00,134
7,products,department_id,int64,0,0.00,21
8,orders,order_id,int64,0,0.00,3421083
9,orders,user_id,int64,0,0.00,206209


### Inspeção visual das tabelas

A exibição das primeiras linhas complementa o resumo do esquema dos dados. Aqui é possível conferir o formato real dos registros, validar nomes de colunas e entender como as chaves se conectam entre as tabelas.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head())

## 5. O que cada tabela representa

- `orders`: pedidos por usuário, com ordem temporal, dia da semana, hora e dias desde o pedido anterior.
- `order_products__prior`: itens de pedidos anteriores. Esta tabela representa o histórico principal de comportamento.
- `order_products__train`: itens de pedidos no conjunto de treino. Pode ser usada como alvo de validação offline.
- `products`: catálogo de produtos, com ligação para `aisles` e `departments`.
- `aisles`: corredores/categorias intermediárias do catálogo.
- `departments`: departamentos principais do catálogo.

Para recomendação, o sinal central é a interação `user_id -> product_id`, obtida ao juntar `orders` com as tabelas `order_products`.

## 6. Qualidade dos dados

Vamos verificar nulos, duplicados e valores fora do domínio esperado.

### Valores ausentes

A primeira verificação de qualidade mede a quantidade e a proporção de valores nulos por coluna. Esse diagnóstico separa ausências esperadas, como o primeiro pedido de cada usuário, de possíveis problemas que precisariam de tratamento.

In [ ]:
missing_summary = (
    schema.loc[schema["nulls"] > 0]
    .sort_values(["null_rate", "nulls"], ascending=False)
    .reset_index(drop=True)
)

missing_summary

### Registros duplicados

A verificação de duplicidade confirma se há linhas repetidas nos arquivos carregados. Duplicatas poderiam distorcer contagens de compra, taxas de recompra e métricas de popularidade dos produtos.

In [ ]:
# Conta linhas duplicadas por tabela usando o validador reutilizável.
duplicate_summary = validator.duplicated_rows_by_table(datasets)

duplicate_summary


### Validação de domínio

Além de nulos e duplicados, algumas colunas possuem faixas de valores esperadas. Esta validação verifica dias da semana, horários, ordem dos pedidos, posição no carrinho e a variável binária de recompra.

In [ ]:
domain_checks = {
    "orders_invalid_order_dow": orders.loc[~orders["order_dow"].between(0, 6)].shape[0],
    "orders_invalid_order_hour": orders.loc[~orders["order_hour_of_day"].between(0, 23)].shape[0],
    "orders_invalid_order_number": orders.loc[orders["order_number"] < 1].shape[0],
    "prior_invalid_add_to_cart_order": order_products_prior.loc[order_products_prior["add_to_cart_order"] < 1].shape[0],
    "train_invalid_add_to_cart_order": order_products_train.loc[order_products_train["add_to_cart_order"] < 1].shape[0],
    "prior_invalid_reordered": order_products_prior.loc[~order_products_prior["reordered"].isin([0, 1])].shape[0],
    "train_invalid_reordered": order_products_train.loc[~order_products_train["reordered"].isin([0, 1])].shape[0],
}

pd.Series(domain_checks, name="invalid_rows").to_frame()

Observação esperada: `days_since_prior_order` deve possuir nulos no primeiro pedido de cada usuário, porque não existe pedido anterior.

In [ ]:
first_orders = orders["order_number"].eq(1)

pd.DataFrame(
    {
        "scenario": ["first_order", "not_first_order"],
        "rows": [first_orders.sum(), (~first_orders).sum()],
        "days_since_prior_order_nulls": [
            orders.loc[first_orders, "days_since_prior_order"].isna().sum(),
            orders.loc[~first_orders, "days_since_prior_order"].isna().sum(),
        ],
    }
)

## 7. Integridade relacional

Aqui validamos se as chaves de uma tabela encontram correspondência nas tabelas de referência.

### Validação das chaves

Como a base é relacional, é importante garantir que produtos, pedidos, corredores e departamentos estejam corretamente referenciados. Falhas nessa etapa poderiam gerar perdas ou inconsistências durante os joins.

In [26]:
integrity_checks = {
    "products_without_aisle": (~products["aisle_id"].isin(aisles["aisle_id"])).sum(),
    "products_without_department": (~products["department_id"].isin(departments["department_id"])).sum(),
    "prior_items_without_order": (~order_products_prior["order_id"].isin(orders["order_id"])).sum(),
    "train_items_without_order": (~order_products_train["order_id"].isin(orders["order_id"])).sum(),
    "prior_items_without_product": (~order_products_prior["product_id"].isin(products["product_id"])).sum(),
    "train_items_without_product": (~order_products_train["product_id"].isin(products["product_id"])).sum(),
}

pd.Series(integrity_checks, name="invalid_references").to_frame()

,invalid_references
products_without_aisle,0
products_without_department,0
prior_items_without_order,0
train_items_without_order,0
prior_items_without_product,0
train_items_without_product,0


## 8. Entendimento das variáveis principais

Com a qualidade e a integridade verificadas, a análise passa a observar as variáveis mais importantes para recomendação: sequência dos pedidos, horário da compra, intervalo entre compras e divisão entre conjuntos de avaliação.

### Estatísticas dos pedidos

As estatísticas descritivas resumem o comportamento temporal dos pedidos. Elas ajudam a observar frequência de compra, distribuição dos horários e possíveis limites naturais das variáveis.

In [27]:
orders[["order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]].describe()

,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3421083.00,3421083.00,3421083.00,3214874.00
mean,17.15,2.78,13.45,11.11
std,17.73,2.05,4.23,9.21
min,1.00,0.00,0.00,0.00
25%,5.00,1.00,10.00,4.00
50%,11.00,3.00,13.00,7.00
75%,23.00,5.00,16.00,15.00
max,100.00,6.00,23.00,30.00


In [28]:
orders["eval_set"].value_counts(normalize=True).rename("rate").to_frame().join(
    orders["eval_set"].value_counts().rename("rows")
)

,rate,rows
eval_set,,
prior,0.94,3214874
train,0.04,131209
test,0.02,75000


### Catálogo enriquecido

O catálogo de produtos é enriquecido com corredor e departamento. Essa visão torna as análises mais interpretáveis, pois substitui parte dos identificadores numéricos por categorias de negócio.

In [29]:
product_catalog = (
    products.merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

product_catalog.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [30]:
department_product_counts = (
    product_catalog["department"]
    .value_counts()
    .rename_axis("department")
    .reset_index(name="products")
)

department_product_counts.head(15)

,department,products
0,personal care,6563
1,snacks,6264
2,pantry,5371
3,beverages,4365
4,frozen,4007
5,dairy eggs,3449
6,household,3085
7,canned goods,2092
8,dry goods pasta,1858
9,produce,1684


## 9. Sinais comportamentais para recomendação

O objetivo desta etapa é responder, com dados reais, quais sinais podem apoiar uma recomendação de produtos. Não estamos criando dados novos nem treinando um modelo aqui. Estamos apenas observando relações existentes na base.

As perguntas principais são:

- Quais produtos o usuário já comprou?
- Quais categorias ele costuma consumir?
- Quais produtos aparecem juntos nos mesmos pedidos?
- Quais produtos usuários parecidos compram?
- Quais produtos são populares dentro dos interesses dele?
- Quais itens ele ainda não comprou, mas têm alta afinidade com seu histórico?

### Construção das interações históricas

Unimos itens comprados, pedidos e catálogo para formar a tabela central de comportamento. Cada linha representa um produto comprado por um usuário em um pedido específico.

In [ ]:
# Une itens comprados, contexto do pedido e catálogo de produtos.
prior_interactions = (
    order_products_prior
    .merge(
        orders[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]],
        on="order_id",
        how="left",
    )
    .merge(product_catalog, on="product_id", how="left")
)

prior_interactions.head()

### Produtos que o usuário já comprou

O histórico individual mostra preferências explícitas do usuário. Ele é útil como sinal de afinidade, mas não deve ser a única fonte de recomendação, pois isso limitaria o sistema a produtos que o usuário já comprou.

In [ ]:
# Escolhe um usuário com bastante histórico para ilustrar as análises comportamentais.
example_user_id = orders["user_id"].value_counts().idxmax()

# Resume quais produtos esse usuário comprou mais vezes.
user_purchase_history = (
    prior_interactions.loc[prior_interactions["user_id"].eq(example_user_id)]
    .groupby(["product_id", "product_name", "aisle", "department"], as_index=False)
    .agg(
        purchase_count=("order_id", "nunique"),
        avg_cart_position=("add_to_cart_order", "mean"),
        last_order_number=("order_number", "max"),
    )
    .sort_values(["purchase_count", "last_order_number"], ascending=False)
)

user_purchase_history.head(15)

### Categorias que o usuário costuma consumir

As categorias favoritas ajudam a recomendar produtos que ainda não foram comprados, mas pertencem a departamentos e corredores com forte presença no histórico do usuário.

In [ ]:
# Agrupa o histórico do usuário por departamento e corredor.
user_category_preferences = (
    prior_interactions.loc[prior_interactions["user_id"].eq(example_user_id)]
    .groupby(["department", "aisle"], as_index=False)
    .agg(items_purchased=("product_id", "count"), unique_products=("product_id", "nunique"))
    .sort_values("items_purchased", ascending=False)
)

user_category_preferences.head(15)

### Produtos que aparecem juntos nos mesmos pedidos

A coocorrência identifica produtos frequentemente comprados no mesmo carrinho. Esse sinal permite sugerir itens complementares, mesmo que o usuário ainda não tenha comprado o produto recomendado.

In [ ]:
# Usa uma amostra de pedidos para estimar produtos comprados juntos sem sobrecarregar a memória.
COOC_SAMPLE_ORDERS = 50_000
TOP_RELATED_PER_PRODUCT = 8

sampled_order_ids = (
    order_products_prior["order_id"]
    .drop_duplicates()
    .sample(n=min(COOC_SAMPLE_ORDERS, order_products_prior["order_id"].nunique()), random_state=42)
)
cooc_source = order_products_prior.loc[
    order_products_prior["order_id"].isin(sampled_order_ids),
    ["order_id", "product_id"],
]

# Conta pares de produtos que aparecem no mesmo pedido.
pair_counts = Counter()
for _, products_in_order in cooc_source.groupby("order_id")["product_id"]:
    basket = sorted(set(products_in_order))

    # Ignora carrinhos muito pequenos ou muito grandes para reduzir ruído e custo.
    if len(basket) < 2 or len(basket) > 30:
        continue

    for left, right in combinations(basket, 2):
        pair_counts[(left, right)] += 1
        pair_counts[(right, left)] += 1

# Junta os IDs ao catálogo para tornar o resultado interpretável.
cooccurrence_sample = (
    pd.DataFrame(
        [(left, right, count) for (left, right), count in pair_counts.items()],
        columns=["product_id", "related_product_id", "cooccurrence_count"],
    )
    .sort_values("cooccurrence_count", ascending=False)
    .head(20)
    .merge(product_catalog[["product_id", "product_name"]], on="product_id", how="left")
    .merge(
        product_catalog[["product_id", "product_name"]].rename(
            columns={"product_id": "related_product_id", "product_name": "related_product_name"}
        ),
        on="related_product_id",
        how="left",
    )
)

cooccurrence_sample

### Produtos comprados por usuários parecidos

Uma forma simples de aproximar usuários semelhantes é comparar suas categorias favoritas. Usuários com o mesmo corredor favorito tendem a compartilhar interesses de consumo.

In [ ]:
# Identifica o corredor favorito de cada usuário pelo número de itens comprados.
user_aisle_counts = (
    prior_interactions.groupby(["user_id", "aisle_id", "aisle"], as_index=False)
    .agg(items_purchased=("product_id", "count"))
)

favorite_aisle_by_user = (
    user_aisle_counts.sort_values(["user_id", "items_purchased"], ascending=[True, False])
    .drop_duplicates("user_id")
    .rename(columns={"aisle_id": "favorite_aisle_id", "aisle": "favorite_aisle"})
    [["user_id", "favorite_aisle_id", "favorite_aisle"]]
)

# Usa o corredor favorito como aproximação simples de usuários semelhantes.
example_favorite_aisle = favorite_aisle_by_user.loc[
    favorite_aisle_by_user["user_id"].eq(example_user_id),
    "favorite_aisle_id",
].iat[0]

similar_user_products = (
    prior_interactions.merge(favorite_aisle_by_user[["user_id", "favorite_aisle_id"]], on="user_id", how="left")
    .query("favorite_aisle_id == @example_favorite_aisle")
    .groupby(["product_id", "product_name", "aisle", "department"], as_index=False)
    .agg(similar_users=("user_id", "nunique"), orders_with_product=("order_id", "nunique"))
    .sort_values(["similar_users", "orders_with_product"], ascending=False)
)

similar_user_products.head(15)

### Produtos populares dentro dos interesses do usuário

Também podemos recomendar produtos populares nos departamentos e corredores favoritos do usuário. Esse sinal amplia a descoberta sem sair das preferências observadas.

In [ ]:
# Seleciona os departamentos favoritos do usuário de exemplo.
example_favorite_departments = (
    prior_interactions.loc[prior_interactions["user_id"].eq(example_user_id)]
    .groupby("department_id")
    .size()
    .sort_values(ascending=False)
    .head(2)
    .index
)

# Busca produtos populares dentro dessas categorias de interesse.
popular_inside_user_interests = (
    prior_interactions.loc[prior_interactions["department_id"].isin(example_favorite_departments)]
    .groupby(["product_id", "product_name", "aisle", "department"], as_index=False)
    .agg(orders_with_product=("order_id", "nunique"), unique_users=("user_id", "nunique"))
    .sort_values(["unique_users", "orders_with_product"], ascending=False)
)

popular_inside_user_interests.head(15)

### Itens novos com alta afinidade

Por fim, removemos produtos já comprados pelo usuário e observamos quais itens novos aparecem com maior força nos sinais de usuários parecidos e categorias favoritas. Esses itens não são dados inventados; são produtos reais que aparecem na base e têm relação com o comportamento observado.

In [ ]:
# Remove produtos já comprados para destacar itens novos que podem ser recomendáveis.
already_purchased = set(user_purchase_history["product_id"])

new_affinity_items = (
    pd.concat(
        [
            similar_user_products.assign(source="similar_users"),
            popular_inside_user_interests.assign(source="favorite_categories"),
        ],
        ignore_index=True,
    )
    .loc[lambda df: ~df["product_id"].isin(already_purchased)]
    .drop_duplicates("product_id")
    .head(20)
)

new_affinity_items[["source", "product_id", "product_name", "aisle", "department"]].head(15)

## 10. Conclusão da análise exploratória

A análise mostra que o dataset oferece sinais suficientes para uma recomendação baseada em comportamento:

- histórico individual de compras;
- preferência por departamentos e corredores;
- produtos comprados juntos no mesmo pedido;
- produtos consumidos por usuários com interesses parecidos;
- produtos populares dentro das categorias favoritas;
- itens novos que ainda não foram comprados pelo usuário, mas têm afinidade com seu histórico.

Com isso, o próximo notebook pode se concentrar em transformar esses sinais em uma base de preparação para modelagem, mantendo o notebook 1 focado apenas em exploração e entendimento dos dados.